# Data Preparation — Cross-Domain Recommendation

CRISP-DM **§3.1–§3.5** implementation. Builds the model-ready inputs the
training loop will consume, per `data_preparation_spec.md` and the §2.x
findings in `notebooks/eda.ipynb`.

**What this notebook produces** under `data/processed/<source>__<target>/`:

- `source_interactions.parquet`, `target_interactions.parquet` — labelled training rows
- `source_item_features.npz`, `target_item_features.npz` — text embeddings + category multi-hot + `has_text`
- `shared_users.parquet` — the cross-vertical mapping supervision set
- `seen_sets.parquet` — for inference-time exclusion
- `id_maps/` — `user_idx` / `item_idx` round-trip tables
- `meta.json` — full run config + embedding details (the CRISP-DM Data Set Description)
- `stats.json` + `STATS.md` — the §3 reporting numbers (per spec §8); the HW2 §3 report is written from these

**Config is exposed at the top.** The source→target pair, the k-core threshold,
and the rating→signal scheme are open decisions per spec §10 — keep them
parameters, not constants. Default pair: Books → Movies_and_TV.


## 0. Setup

In [1]:
from __future__ import annotations
import json, os, hashlib, random, time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.csv as pacsv
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/Users/orimood/Desktop/homework/Amazon_CR')
RAW = PROJECT_ROOT / 'data' / 'raw'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

# embedding cache lives outside any single pair so different pairs reuse it
EMBED_CACHE = PROJECT_ROOT / 'data' / 'embed_cache'
EMBED_CACHE.mkdir(parents=True, exist_ok=True)


### Config — spec §2

Single dataclass. Defaults match the spec; the open decisions per spec §10
(domain pair, k-core, rating-to-signal scheme, split mode) are exposed as
parameters.

`embed_sample_size = None` runs the full embedding job over all kept items
(~700k for the default pair). Set to an int (e.g. 5000) for a fast smoke run
during pipeline development — placeholder zero vectors are written for the
rest.


In [2]:
@dataclass
class Config:
    # domain pair (spec §2 — NOT locked; see spec §10)
    source: str = 'Books'
    target: str = 'Movies_and_TV'

    # density filter
    k_core: int = 5
    k_core_overrides: Dict[str, int] = field(default_factory=dict)  # e.g. {'target': 3}

    # rating -> training signal (spec §3.3; HW2 §2.1 contemplated scheme)
    # Sampled negatives are PRECOMPUTED and frozen into the interaction tables
    # for a fixed inspectable dataset (spec §3.3(a) default).
    positive_threshold: int = 4
    drop_ratings: Tuple[int, ...] = (0, 3)
    explicit_negative: Tuple[int, ...] = (1, 2)
    neg_sample_ratio: int = 4
    explicit_neg_weight: float = 1.0

    # text embeddings (spec §3.3(c))
    embed_model: str = 'sentence-transformers/all-MiniLM-L6-v2'
    embed_dim: int = 384
    max_seq_length: int = 256              # explicit truncation (MiniLM default)
    embed_normalize: bool = True           # recorded in meta.json
    text_fields: Tuple[str, ...] = ('title', 'description', 'features')
    text_fallback: Tuple[str, ...] = ('title', 'store', 'details')
    embed_batch: int = 256
    embed_sample_size: Optional[int] = 0  # set to None for full embedding run  # None = full run; int = smoke mode
    embed_device: str = 'auto'              # 'auto' | 'mps' | 'cuda' | 'cpu'

    # splits
    split: str = 'temporal'                 # 'temporal' (primary) | 'random'
    split_ratios: Tuple[float, float, float] = (0.8, 0.1, 0.1)
    seed: int = 42

    # quality floor — spec §5
    overlap_floor: int = 10_000

CFG = Config()
np.random.seed(CFG.seed); random.seed(CFG.seed)
print(json.dumps(asdict(CFG), indent=2, default=str))

PAIR_DIR = PROCESSED / f'{CFG.source}__{CFG.target}'
PAIR_DIR.mkdir(parents=True, exist_ok=True)
print(f'\noutput dir: {PAIR_DIR}')


{
  "source": "Books",
  "target": "Movies_and_TV",
  "k_core": 5,
  "k_core_overrides": {},
  "positive_threshold": 4,
  "drop_ratings": [
    0,
    3
  ],
  "explicit_negative": [
    1,
    2
  ],
  "neg_sample_ratio": 4,
  "explicit_neg_weight": 1.0,
  "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
  "embed_dim": 384,
  "max_seq_length": 256,
  "embed_normalize": true,
  "text_fields": [
    "title",
    "description",
    "features"
  ],
  "text_fallback": [
    "title",
    "store",
    "details"
  ],
  "embed_batch": 256,
  "embed_sample_size": 0,
  "embed_device": "auto",
  "split": "temporal",
  "split_ratios": [
    0.8,
    0.1,
    0.1
  ],
  "seed": 42,
  "overlap_floor": 10000
}

output dir: /Users/orimood/Desktop/homework/Amazon_CR/data/processed/Books__Movies_and_TV


In [3]:
# Two output dicts. RUN_META → meta.json (config + embedding details).
# STATS → stats.json (per spec §8, the §3 reporting numbers).
RUN_META: Dict = {
    'config': asdict(CFG),
    'snapshot_date': time.strftime('%Y-%m-%d'),
    'embed': {
        'model': CFG.embed_model,
        'dim': CFG.embed_dim,
        'max_seq_length': CFG.max_seq_length,
        'normalize': CFG.embed_normalize,
        'cache_dir': str(EMBED_CACHE),
    },
}

STATS: Dict = {
    'pair': f'{CFG.source}__{CFG.target}',
    'snapshot_date': time.strftime('%Y-%m-%d'),
    'config': asdict(CFG),
    'select': {},
    'clean': {},
    'construct': {},
    'integrate': {},
    'format': {},
}

def log_stage(name: str, **kv):
    print(f'[{name}]')
    for k, v in kv.items():
        if isinstance(v, dict):
            print(f'  {k}:')
            for kk, vv in v.items():
                print(f'    {kk}: {vv}')
        else:
            print(f'  {k}: {v}')


## §3.1 Select Data — *Rationale for Inclusion / Exclusion*

Operations in spec-mandated order:

1. Drop unused columns at read time (handled by `pyarrow.csv` schema).
2. Drop invalid ratings — `0` (invalid scan, four Books rows) and `3`
   (ambiguous). **Counted separately** per spec §8. The 3-star rows still
   appear in the per-user **seen set** built in §3.3.
3. Iterative k-core on the remaining {1,2,4,5}-rated rows.
4. Compute cross-vertical user overlap; assert ≥ `overlap_floor` (spec §5).


In [4]:
def load_ratings(vertical: str) -> pd.DataFrame:
    p = RAW / 'reviews' / f'{vertical}.csv'
    tbl = pacsv.read_csv(
        p,
        convert_options=pacsv.ConvertOptions(column_types={
            'user_id': 'string', 'parent_asin': 'string',
            'rating': 'float32',     # CSV stores rating as '5.0'; cast below
            'timestamp': 'int64',
        })
    )
    df = tbl.to_pandas()
    df['rating'] = df['rating'].astype('int8')
    print(f'  loaded {vertical:18s} rows={len(df):>12,d}')
    return df

print('Loading source + target ratings only (other verticals stay on disk).')
src = load_ratings(CFG.source)
tgt = load_ratings(CFG.target)


Loading source + target ratings only (other verticals stay on disk).


  loaded Books              rows=  29,139,329


  loaded Movies_and_TV      rows=  17,158,519


In [5]:
# spec §3.1: 'recompute the per-user seen set from the unfiltered interactions separately'
# stash the full ratings (any value, incl. 0/3) for seen-set construction in §3.3
src_all = src.copy()
tgt_all = tgt.copy()

# capture 0-core baseline numbers per spec §8
def baseline(df: pd.DataFrame) -> Dict:
    return {
        'interactions_0core': int(len(df)),
        'users_0core': int(df['user_id'].nunique()),
        'items_0core': int(df['parent_asin'].nunique()),
    }

STATS['select']['source'] = {'vertical': CFG.source, **baseline(src)}
STATS['select']['target'] = {'vertical': CFG.target, **baseline(tgt)}

# drop training-invalid ratings — count rating==0 and rating==3 SEPARATELY (§8)
def drop_invalid(df: pd.DataFrame, name: str) -> Tuple[pd.DataFrame, Dict]:
    n_in = len(df)
    n_drop_0 = int((df['rating'] == 0).sum())
    n_drop_3 = int((df['rating'] == 3).sum())
    df_out = df[~df['rating'].isin(CFG.drop_ratings)].reset_index(drop=True)
    counts = {'dropped_rating_0': n_drop_0, 'dropped_rating_3': n_drop_3,
              'interactions_after_drop': int(len(df_out))}
    print(f'  {name}: rating==0 dropped {n_drop_0:>11,d}  '
          f'rating==3 dropped {n_drop_3:>11,d}  remaining {len(df_out):>12,d}')
    return df_out, counts

src, src_drop_counts = drop_invalid(src, 'source')
tgt, tgt_drop_counts = drop_invalid(tgt, 'target')
STATS['select']['source'].update(src_drop_counts)
STATS['select']['target'].update(tgt_drop_counts)


  source: rating==0 dropped           4  rating==3 dropped   2,032,688  remaining   27,106,637


  target: rating==0 dropped           0  rating==3 dropped   1,248,470  remaining   15,910,049


In [6]:
def kcore_filter(df: pd.DataFrame, k: int, max_iters: int = 30) -> pd.DataFrame:
    prev = -1
    for _ in range(max_iters):
        u = df.groupby('user_id').size()
        df = df[df['user_id'].isin(u[u >= k].index)]
        i = df.groupby('parent_asin').size()
        df = df[df['parent_asin'].isin(i[i >= k].index)]
        if len(df) == prev:
            return df.reset_index(drop=True)
        prev = len(df)
    return df.reset_index(drop=True)

k_src = CFG.k_core_overrides.get('source', CFG.k_core)
k_tgt = CFG.k_core_overrides.get('target', CFG.k_core)

t = time.time()
src_5c = kcore_filter(src, k_src)
print(f'  source 5-core: rows {len(src):>12,d} -> {len(src_5c):>12,d}  ({time.time()-t:.1f}s)')
t = time.time()
tgt_5c = kcore_filter(tgt, k_tgt)
print(f'  target 5-core: rows {len(tgt):>12,d} -> {len(tgt_5c):>12,d}  ({time.time()-t:.1f}s)')

def post_5core(df_5c: pd.DataFrame, baseline_0core: int, k: int) -> Dict:
    return {
        'k_core': int(k),
        'interactions_5core': int(len(df_5c)),
        'users_5core': int(df_5c['user_id'].nunique()),
        'items_5core': int(df_5c['parent_asin'].nunique()),
        'retention_pct_vs_0core': round(len(df_5c) / baseline_0core * 100, 2),
    }

STATS['select']['source'].update(post_5core(
    src_5c, STATS['select']['source']['interactions_0core'], k_src))
STATS['select']['target'].update(post_5core(
    tgt_5c, STATS['select']['target']['interactions_0core'], k_tgt))


  source 5-core: rows   27,106,637 ->    8,130,341  (62.2s)


  target 5-core: rows   15,910,049 ->    6,486,790  (30.2s)


In [7]:
# spec §3.4 / §5 — cross-vertical user intersection, with overlap floor assertion
src_users = set(src_5c['user_id'].unique())
tgt_users = set(tgt_5c['user_id'].unique())
shared_users = src_users & tgt_users

print(f'  users in source (k={k_src}-core): {len(src_users):,}')
print(f'  users in target (k={k_tgt}-core): {len(tgt_users):,}')
print(f'  shared users                    : {len(shared_users):,}')
print(f'  Jaccard                         : {len(shared_users) / len(src_users | tgt_users):.4f}')

if len(shared_users) < CFG.overlap_floor:
    print(f'\n  WARN: shared-user count below floor {CFG.overlap_floor:,} '
          f'-- consider k_core_overrides (HW1 R1 contingency)')
else:
    print(f'\n  OK: shared users clear floor {CFG.overlap_floor:,}')

log_stage('3.1_select',
          source=STATS['select']['source'], target=STATS['select']['target'],
          shared_users=len(shared_users))


  users in source (k=5-core): 687,023
  users in target (k=5-core): 592,843
  shared users                    : 109,206
  Jaccard                         : 0.0933

  OK: shared users clear floor 10,000
[3.1_select]
  source:
    vertical: Books
    interactions_0core: 29139329
    users_0core: 10297355
    items_0core: 4446065
    dropped_rating_0: 4
    dropped_rating_3: 2032688
    interactions_after_drop: 27106637
    k_core: 5
    interactions_5core: 8130341
    users_5core: 687023
    items_5core: 440134
    retention_pct_vs_0core: 27.9
  target:
    vertical: Movies_and_TV
    interactions_0core: 17158519
    users_0core: 6503429
    items_0core: 747764
    dropped_rating_0: 0
    dropped_rating_3: 1248470
    interactions_after_drop: 15910049
    k_core: 5
    interactions_5core: 6486790
    users_5core: 592843
    items_5core: 181532
    retention_pct_vs_0core: 37.81
  shared_users: 109206


## §3.2 Clean Data — *Data Cleaning Report*

Stream the metadata for each vertical, keeping only items that survived
selection. Build the per-item text using the **fallback chain** required by
the spec — critical because §2.4 showed 42% of Movies_and_TV items lack a
`title` and 54% of Movies interactions land on those items (most of them
salvageable via Directors / Producers / Starring inside `details`).

Cleaning also:
- normalizes `store` and `categories` strings (lowercase + strip + collapse ws);
- tags every item with its file-of-origin vertical (`main_category` is the
  live store label, **not** the partition — spec §3.2);
- tracks per-field missing-value rates on the kept set (spec §8);
- counts items that fell back to the secondary text chain vs items with no
  usable text at all (spec §8);
- asserts the §5 guards on the kept set.


In [8]:
def _normalize(s: Optional[str]) -> Optional[str]:
    if not isinstance(s, str):
        return None
    s = ' '.join(s.split())
    return s if s else None

def _join_list(xs) -> str:
    if not isinstance(xs, list):
        return ''
    parts = []
    for x in xs:
        if isinstance(x, str) and x.strip():
            parts.append(x.strip())
    return '\n'.join(parts)

def _details_to_text(d) -> str:
    """Flatten the details dict into prose-ish text.

    For Movies_and_TV (per §2.4 finding) this is where Directors / Producers /
    Starring live for the 53.9% of interactions on untitled items."""
    if not isinstance(d, dict):
        return ''
    parts = []
    for k, v in d.items():
        if isinstance(v, list):
            v_str = ', '.join(x for x in v if isinstance(x, str))
        elif isinstance(v, str):
            v_str = v
        else:
            v_str = str(v)
        if v_str:
            parts.append(f'{k}: {v_str}')
    return '. '.join(parts)

def build_text(rec: Dict, text_fields: Tuple[str, ...], text_fallback: Tuple[str, ...]
              ) -> Tuple[str, str]:
    """Return (item_text, source) where source ∈ {'primary','fallback','none'}."""
    # primary: title + description + features
    chunks = []
    if 'title' in text_fields and rec.get('title'):
        chunks.append(_normalize(rec['title']))
    if 'description' in text_fields:
        d = _join_list(rec.get('description'))
        if d: chunks.append(d)
    if 'features' in text_fields:
        f = _join_list(rec.get('features'))
        if f: chunks.append(f)
    primary = '\n'.join(c for c in chunks if c)
    if primary:
        return primary, 'primary'

    # fallback: title + store + details (synthesised pseudo-title)
    chunks = []
    if 'title' in text_fallback and rec.get('title'):
        chunks.append(_normalize(rec['title']))
    if 'store' in text_fallback and _normalize(rec.get('store')):
        chunks.append(_normalize(rec['store']))
    if 'details' in text_fallback:
        d = _details_to_text(rec.get('details'))
        if d: chunks.append(d)
    fallback = '. '.join(c for c in chunks if c)
    if fallback:
        return fallback, 'fallback'
    return '', 'none'


In [9]:
# fields we track for per-field missing-value rates on the kept set (spec §8)
TRACKED_FIELDS = ['title', 'description', 'features', 'categories',
                  'main_category', 'store', 'details', 'price',
                  'average_rating', 'rating_number', 'images', 'author']

def stream_meta_for_kept(vertical: str, kept_asins: Set[str]) -> Tuple[pd.DataFrame, Dict]:
    """One pass over a vertical's meta JSONL; build the item table for kept asins
    AND collect the per-field missing-rate stats spec §8 wants."""
    p = RAW / 'meta' / f'meta_{vertical}.jsonl'
    rows = []
    n_total = 0
    n_kept = 0
    n_primary = 0
    n_fallback = 0
    n_none = 0
    populated = Counter()
    with open(p) as f:
        for line in tqdm(f, desc=f'meta {vertical}', unit=' rec'):
            n_total += 1
            r = json.loads(line)
            a = r.get('parent_asin')
            if a not in kept_asins:
                continue
            n_kept += 1
            # per-field populated check (for missing-rate report)
            for fld in TRACKED_FIELDS:
                v = r.get(fld)
                if v is None: continue
                if isinstance(v, list) and len(v) == 0: continue
                if isinstance(v, dict) and len(v) == 0: continue
                if isinstance(v, str) and not v.strip(): continue
                populated[fld] += 1
            text, src_tag = build_text(r, CFG.text_fields, CFG.text_fallback)
            if src_tag == 'primary': n_primary += 1
            elif src_tag == 'fallback': n_fallback += 1
            else: n_none += 1
            cats = r.get('categories') or []
            rows.append({
                'parent_asin': a,
                'vertical': vertical,                              # spec §3.2
                'item_text': text,
                'has_text': src_tag != 'none',
                'text_source': src_tag,                            # primary | fallback | none
                'categories': [_normalize(c) for c in cats if isinstance(c, str)],
                'main_category': _normalize(r.get('main_category')),
                'store_norm': _normalize(r.get('store')),
            })
    items = pd.DataFrame(rows)
    stats = {
        'meta_streamed': n_total,
        'items_kept': n_kept,
        'text_primary': n_primary,
        'text_fallback': n_fallback,
        'text_none_placeholder': n_none,
        'missing_value_rates': {k: round(1 - populated.get(k, 0) / max(1, n_kept), 4)
                                for k in TRACKED_FIELDS},
    }
    return items, stats

src_asins_5c = set(src_5c['parent_asin'].unique())
tgt_asins_5c = set(tgt_5c['parent_asin'].unique())

t = time.time()
src_items, src_clean_stats = stream_meta_for_kept(CFG.source, src_asins_5c)
print(f'  source: kept {src_clean_stats["items_kept"]:,}  '
      f'primary={src_clean_stats["text_primary"]:,}  '
      f'fallback={src_clean_stats["text_fallback"]:,}  '
      f'none={src_clean_stats["text_none_placeholder"]:,}  ({time.time()-t:.1f}s)')

t = time.time()
tgt_items, tgt_clean_stats = stream_meta_for_kept(CFG.target, tgt_asins_5c)
print(f'  target: kept {tgt_clean_stats["items_kept"]:,}  '
      f'primary={tgt_clean_stats["text_primary"]:,}  '
      f'fallback={tgt_clean_stats["text_fallback"]:,}  '
      f'none={tgt_clean_stats["text_none_placeholder"]:,}  ({time.time()-t:.1f}s)')


meta Books: 0 rec [00:00, ? rec/s]

  source: kept 440,134  primary=440,134  fallback=0  none=0  (53.2s)


meta Movies_and_TV: 0 rec [00:00, ? rec/s]

  target: kept 181,532  primary=106,431  fallback=75,097  none=4  (6.9s)


In [10]:
# spec §5 — runtime quality guards on the kept set; collect counts for spec §8
def assert_quality(ratings: pd.DataFrame, items: pd.DataFrame, name: str
                  ) -> Tuple[pd.DataFrame, Dict]:
    # ratings hygiene
    assert ratings['user_id'].notna().all(), f'{name}: null user_id'
    assert ratings['parent_asin'].notna().all(), f'{name}: null parent_asin'
    assert ratings['rating'].between(1, 5).all(), f'{name}: rating out of [1,5]'
    ts = pd.to_datetime(ratings['timestamp'], unit='ms')
    assert (ts >= pd.Timestamp('1996-01-01')).all() and (ts < pd.Timestamp('2024-01-01')).all(), \
        f'{name}: timestamps out of expected window'
    # ratings ⨝ meta -> orphans
    item_set = set(items['parent_asin'])
    n_orphan = int((~ratings['parent_asin'].isin(item_set)).sum())
    assert n_orphan == 0, f'{name}: {n_orphan} orphan interactions'
    # duplicate (user, item) pairs — dedupe with latest timestamp wins
    n_dup = int(ratings.duplicated(subset=['user_id', 'parent_asin']).sum())
    if n_dup:
        print(f'  {name}: deduping {n_dup:,} duplicate (user, item) pairs '
              f'(latest timestamp wins)')
        idx = ratings.sort_values('timestamp').groupby(['user_id', 'parent_asin']).tail(1).index
        ratings = ratings.loc[idx].reset_index(drop=True)
    return ratings, {'orphan_interactions': n_orphan, 'duplicate_pairs': n_dup}

src_5c, src_q = assert_quality(src_5c, src_items, 'source')
tgt_5c, tgt_q = assert_quality(tgt_5c, tgt_items, 'target')
print('  all §5 guards passed.')

STATS['clean']['source'] = {**src_clean_stats, **src_q}
STATS['clean']['target'] = {**tgt_clean_stats, **tgt_q}
log_stage('3.2_clean',
          source=STATS['clean']['source'], target=STATS['clean']['target'])


  all §5 guards passed.
[3.2_clean]
  source:
    meta_streamed: 4448181
    items_kept: 440134
    text_primary: 440134
    text_fallback: 0
    text_none_placeholder: 0
    missing_value_rates: {'title': 0.0, 'description': 0.2619, 'features': 0.0183, 'categories': 0.0171, 'main_category': 0.0006, 'store': 0.0047, 'details': 0.0414, 'price': 0.1131, 'average_rating': 0.0, 'rating_number': 0.0, 'images': 0.1131, 'author': 0.1153}
    orphan_interactions: 0
    duplicate_pairs: 0
  target:
    meta_streamed: 748224
    items_kept: 181532
    text_primary: 106431
    text_fallback: 75097
    text_none_placeholder: 4
    missing_value_rates: {'title': 0.4137, 'description': 0.4569, 'features': 0.9456, 'categories': 0.4142, 'main_category': 0.0127, 'store': 0.4679, 'details': 0.0051, 'price': 0.5308, 'average_rating': 0.0, 'rating_number': 0.0, 'images': 0.0, 'author': 1.0}
    orphan_interactions: 0
    duplicate_pairs: 0


## §3.3 Construct Data — *Derived Attributes + Generated Records*

Four derived artifacts, in order:

- **(a) Labels** — `≥4 → positive`, `1/2 → explicit negative`, `3 → dropped`
  (already removed in §3.1; the per-user **seen set** is built from the
  *unfiltered* ratings, so 3s still influence inference-time exclusion).
  Each positive gets `neg_sample_ratio` **precomputed** sampled-unobserved
  negatives drawn from the catalog (excluding seen items, seeded with `CFG.seed`
  for reproducibility per spec §3.3(a) default).
- **(b) Seen-set vs positive-set** — two distinct per-user sets, both persisted.
- **(c) Item text embeddings** — `embed_model` over the `item_text` from §3.2,
  with **explicit truncation** to `CFG.max_seq_length` word-pieces (spec §3.3(c)),
  cached by content hash. Items with `has_text == False` get a zero vector
  (the *generated record* placeholder). Outputs are L2-normalized when
  `embed_normalize == True` (recorded in `meta.json`).
- **(d) Category multi-hot** — sparse, supports the categories-only ablation.


### 3.3(a) Labels — positive set + explicit negatives

In [11]:
def build_labels(df: pd.DataFrame, name: str) -> pd.DataFrame:
    df = df.copy()
    df['label'] = -1
    df.loc[df['rating'] >= CFG.positive_threshold, 'label'] = 1
    df.loc[df['rating'].isin(CFG.explicit_negative), 'label'] = 0
    assert (df['label'] != -1).all(), f'{name}: unlabelled rows'
    df['weight'] = np.where(df['label'] == 0, CFG.explicit_neg_weight, 1.0).astype('float32')
    n_pos = int((df['label'] == 1).sum())
    n_neg = int((df['label'] == 0).sum())
    print(f'  {name}: positives={n_pos:,}  explicit_neg={n_neg:,}  '
          f'pos_ratio={n_pos/(n_pos+n_neg):.3f}')
    return df

src_lab = build_labels(src_5c, 'source')
tgt_lab = build_labels(tgt_5c, 'target')


  source: positives=7,563,762  explicit_neg=566,579  pos_ratio=0.930


  target: positives=5,713,713  explicit_neg=773,077  pos_ratio=0.881


### 3.3(b) Seen sets — built from the unfiltered ratings

In [12]:
# the seen set spans every interaction the user ever had in that vertical,
# including the 3s we dropped from labels (spec §3.3(b))
def build_seen(df_all: pd.DataFrame, name: str) -> Dict[str, Set[str]]:
    seen = defaultdict(set)
    for u, a in tqdm(zip(df_all['user_id'], df_all['parent_asin']),
                     total=len(df_all), desc=f'seen {name}'):
        seen[u].add(a)
    return seen

src_seen = build_seen(src_all, 'source')
tgt_seen = build_seen(tgt_all, 'target')
print(f'  source seen-set users: {len(src_seen):,}  '
      f'mean items/user: {np.mean([len(s) for s in src_seen.values()]):.2f}')
print(f'  target seen-set users: {len(tgt_seen):,}  '
      f'mean items/user: {np.mean([len(s) for s in tgt_seen.values()]):.2f}')


seen source:   0%|          | 0/29139329 [00:00<?, ?it/s]

seen target:   0%|          | 0/17158519 [00:00<?, ?it/s]

  source seen-set users: 10,297,355  mean items/user: 2.83


  target seen-set users: 6,503,429  mean items/user: 2.64


### 3.3(a, continued) — sampled negatives from the unobserved catalog

In [13]:
def sample_negatives(positives: pd.DataFrame, seen: Dict[str, Set[str]],
                     all_items: List[str], k: int, name: str) -> pd.DataFrame:
    """For each positive (u, i), draw k items the user has not interacted with.

    Spec §3.3(a): seeded sample, uniform over the *unobserved* catalog,
    excluding the user's full seen set (which includes 3-star rated items and
    1/2-star items). Negatives are precomputed and frozen into the interaction
    tables.
    """
    rng = np.random.default_rng(CFG.seed)
    item_arr = np.array(all_items)
    cat_size = len(item_arr)
    rows = []
    pos_users = positives['user_id'].values
    pos_ts = positives['timestamp'].values
    for u, t_ in tqdm(zip(pos_users, pos_ts), total=len(positives), desc=f'sample_neg {name}'):
        forbidden = seen.get(u, set())
        drawn = []
        tries = 0
        while len(drawn) < k and tries < k * 8:
            cand = item_arr[rng.integers(0, cat_size, size=k * 2)]
            for c in cand:
                if c not in forbidden and c not in drawn:
                    drawn.append(c)
                    if len(drawn) == k:
                        break
            tries += 1
        for c in drawn:
            rows.append((u, c, t_))
    out = pd.DataFrame(rows, columns=['user_id', 'parent_asin', 'timestamp'])
    out['rating'] = 0  # sentinel
    out['label'] = 0
    out['weight'] = 1.0
    return out

src_catalog = src_items['parent_asin'].tolist()
tgt_catalog = tgt_items['parent_asin'].tolist()

src_pos = src_lab[src_lab['label'] == 1]
tgt_pos = tgt_lab[tgt_lab['label'] == 1]

t = time.time()
src_sampled_neg = sample_negatives(src_pos, src_seen, src_catalog, CFG.neg_sample_ratio, 'source')
print(f'  source sampled negatives: {len(src_sampled_neg):,}  ({time.time()-t:.1f}s)')
t = time.time()
tgt_sampled_neg = sample_negatives(tgt_pos, tgt_seen, tgt_catalog, CFG.neg_sample_ratio, 'target')
print(f'  target sampled negatives: {len(tgt_sampled_neg):,}  ({time.time()-t:.1f}s)')


sample_neg source:   0%|          | 0/7563762 [00:00<?, ?it/s]

  source sampled negatives: 30,255,048  (82.7s)


sample_neg target:   0%|          | 0/5713713 [00:00<?, ?it/s]

  target sampled negatives: 22,854,852  (62.5s)


In [14]:
# combine: positives + explicit negatives + sampled negatives -> the modeling table
def combine_signals(labelled: pd.DataFrame, sampled_neg: pd.DataFrame) -> pd.DataFrame:
    keep = ['user_id', 'parent_asin', 'rating', 'timestamp', 'label', 'weight']
    return pd.concat([labelled[keep], sampled_neg[keep]], ignore_index=True)

src_signal = combine_signals(src_lab, src_sampled_neg)
tgt_signal = combine_signals(tgt_lab, tgt_sampled_neg)
print(f'  source signal rows: {len(src_signal):,}  '
      f'(pos {(src_signal.label==1).sum():,} / explicit-neg {(src_signal.rating.isin(CFG.explicit_negative)).sum():,} '
      f'/ sampled-neg {(src_signal.rating==0).sum():,})')
print(f'  target signal rows: {len(tgt_signal):,}  '
      f'(pos {(tgt_signal.label==1).sum():,} / explicit-neg {(tgt_signal.rating.isin(CFG.explicit_negative)).sum():,} '
      f'/ sampled-neg {(tgt_signal.rating==0).sum():,})')

# stash label distribution for STATS
def label_dist(lab: pd.DataFrame, sampled_neg: pd.DataFrame, drop_3_count: int) -> Dict:
    return {
        'positives': int((lab['label'] == 1).sum()),
        'explicit_negatives': int((lab['label'] == 0).sum()),
        'dropped_3': int(drop_3_count),
        'sampled_negatives': int(len(sampled_neg)),
    }

STATS['construct']['source'] = {
    'label_distribution': label_dist(
        src_lab, src_sampled_neg,
        STATS['select']['source']['dropped_rating_3']),
}
STATS['construct']['target'] = {
    'label_distribution': label_dist(
        tgt_lab, tgt_sampled_neg,
        STATS['select']['target']['dropped_rating_3']),
}


  source signal rows: 38,385,389  (pos 7,563,762 / explicit-neg 566,579 / sampled-neg 30,255,048)


  target signal rows: 29,341,642  (pos 5,713,713 / explicit-neg 773,077 / sampled-neg 22,854,852)


### 3.3(c) Item text embeddings — `all-MiniLM-L6-v2`, explicit truncation, cached

Encoding ~700k items is the heaviest job in the pipeline (spec §7, HW1 risk R2).
Three policies make it manageable:

- **Explicit `model.max_seq_length = CFG.max_seq_length`** (spec §3.3(c)) — texts
  beyond 256 word-pieces are truncated by the tokenizer.
- **Cache by `sha1(item_text)`** — re-runs and ablations skip already-encoded text.
- **Sample mode** — `CFG.embed_sample_size = int(N)` encodes only the first N
  unique texts for fast smoke tests; the rest get the zero placeholder. The
  production run leaves it at `None`.

Items with `has_text == False` always get the zero placeholder.


In [15]:
def pick_device(want: str) -> str:
    if want != 'auto':
        return want
    try:
        import torch
        if torch.backends.mps.is_available():
            return 'mps'
        if torch.cuda.is_available():
            return 'cuda'
    except ImportError:
        pass
    return 'cpu'

def text_hash(s: str) -> str:
    return hashlib.sha1(s.encode('utf-8')).hexdigest()

def encode_items(items: pd.DataFrame, name: str) -> Tuple[np.ndarray, Dict]:
    """Returns (emb, stats) where stats has counts spec §8 wants."""
    texts = items['item_text'].tolist()
    has_text = items['has_text'].values

    # cache layer — keyed by sha1(text)
    cache_path = EMBED_CACHE / f'{CFG.embed_model.replace("/", "_")}.npz'
    cache: Dict[str, np.ndarray] = {}
    if cache_path.exists():
        z = np.load(cache_path)
        cache = {k: z[k] for k in z.files}
        print(f'  loaded {len(cache):,} cached vectors from {cache_path.name}')

    # decide which need encoding
    hashes = [text_hash(t) if t else '' for t in texts]
    cache_hits = 0
    need_idx, need_text = [], []
    for i, (t, h) in enumerate(zip(texts, hashes)):
        if not t or not has_text[i]:
            continue
        if h in cache:
            cache_hits += 1
            continue
        need_idx.append(i)
        need_text.append(t)

    n_truncated = 0  # filled in below
    if CFG.embed_sample_size is not None and len(need_text) > CFG.embed_sample_size:
        print(f'  sample mode: encoding only first {CFG.embed_sample_size:,} of {len(need_text):,} needed')
        need_idx = need_idx[:CFG.embed_sample_size]
        need_text = need_text[:CFG.embed_sample_size]

    if need_text:
        from sentence_transformers import SentenceTransformer
        device = pick_device(CFG.embed_device)
        print(f'  encoding {len(need_text):,} texts on device={device} '
              f'(model={CFG.embed_model}, max_seq_length={CFG.max_seq_length})')
        model = SentenceTransformer(CFG.embed_model, device=device)
        # spec §3.3(c): explicit truncation
        model.max_seq_length = CFG.max_seq_length
        # count how many texts will actually be truncated by the tokenizer
        tok = model.tokenizer
        for t_ in need_text:
            if len(tok.encode(t_, add_special_tokens=False)) > CFG.max_seq_length:
                n_truncated += 1
        new_emb = model.encode(
            need_text, batch_size=CFG.embed_batch, show_progress_bar=True,
            convert_to_numpy=True, normalize_embeddings=CFG.embed_normalize,
        )
        for k, i in enumerate(need_idx):
            cache[hashes[i]] = new_emb[k]
        np.savez(cache_path, **cache)
        print(f'  cache now holds {len(cache):,} vectors  ({n_truncated:,} of '
              f'{len(need_text):,} new texts exceeded max_seq_length and were truncated)')

    # assemble final aligned matrix
    emb = np.zeros((len(items), CFG.embed_dim), dtype='float32')
    n_placeholder = 0
    for i, h in enumerate(hashes):
        if h and h in cache:
            emb[i] = cache[h]
        else:
            n_placeholder += 1
    print(f'  {name}: encoded items={len(emb):,}  placeholder (zero) rows={n_placeholder:,}  '
          f'cache_hits={cache_hits:,}')
    stats = {
        'items_total': int(len(emb)),
        'items_embedded': int(len(emb) - n_placeholder),
        'items_placeholder': int(n_placeholder),
        'cache_hits': int(cache_hits),
        'texts_truncated_this_run': int(n_truncated),
    }
    return emb, stats

t = time.time()
src_emb, src_emb_stats = encode_items(src_items, 'source')
print(f'  source embed time: {time.time()-t:.1f}s')
t = time.time()
tgt_emb, tgt_emb_stats = encode_items(tgt_items, 'target')
print(f'  target embed time: {time.time()-t:.1f}s')

STATS['construct']['source']['embedding'] = src_emb_stats
STATS['construct']['target']['embedding'] = tgt_emb_stats


  loaded 9,963 cached vectors from sentence-transformers_all-MiniLM-L6-v2.npz


  sample mode: encoding only first 0 of 435,094 needed
  source: encoded items=440,134  placeholder (zero) rows=435,094  cache_hits=5,040
  source embed time: 6.0s


  loaded 9,963 cached vectors from sentence-transformers_all-MiniLM-L6-v2.npz
  sample mode: encoding only first 0 of 176,186 needed


  target: encoded items=181,532  placeholder (zero) rows=176,190  cache_hits=5,342
  target embed time: 9.6s


### 3.3(d) Category multi-hot — supports categories-only ablation

Shared vocabulary of normalized category strings across both verticals. In
practice the two verticals' taxonomies barely overlap, but the matrix is the
right shape and the model decides how to use it.


In [16]:
from scipy.sparse import csr_matrix

def build_category_vocab(*item_dfs: pd.DataFrame) -> Dict[str, int]:
    vocab: Dict[str, int] = {}
    for df in item_dfs:
        for cats in df['categories']:
            for c in cats:
                if c and c not in vocab:
                    vocab[c] = len(vocab)
    return vocab

def items_to_multihot(items: pd.DataFrame, vocab: Dict[str, int]) -> csr_matrix:
    rows, cols = [], []
    for i, cats in enumerate(items['categories']):
        for c in cats:
            j = vocab.get(c)
            if j is not None:
                rows.append(i); cols.append(j)
    data = np.ones(len(rows), dtype='float32')
    return csr_matrix((data, (rows, cols)), shape=(len(items), len(vocab)))

CAT_VOCAB = build_category_vocab(src_items, tgt_items)
src_cat = items_to_multihot(src_items, CAT_VOCAB)
tgt_cat = items_to_multihot(tgt_items, CAT_VOCAB)

# distinct category PATHS (tuple-equal) per vertical — spec §8 asks for this
src_distinct_paths = len({tuple(p) for p in src_items['categories']})
tgt_distinct_paths = len({tuple(p) for p in tgt_items['categories']})

print(f'  category vocab size : {len(CAT_VOCAB):,}')
print(f'  source distinct paths: {src_distinct_paths:,}  category mat: {src_cat.shape} nnz={src_cat.nnz:,}')
print(f'  target distinct paths: {tgt_distinct_paths:,}  category mat: {tgt_cat.shape} nnz={tgt_cat.nnz:,}')

STATS['construct']['category_vocab_size'] = int(len(CAT_VOCAB))
STATS['construct']['source']['category_distinct_paths'] = int(src_distinct_paths)
STATS['construct']['target']['category_distinct_paths'] = int(tgt_distinct_paths)

log_stage('3.3_construct',
          src_emb=src_emb_stats, tgt_emb=tgt_emb_stats,
          category_vocab=len(CAT_VOCAB))


  category vocab size : 1,661
  source distinct paths: 1,216  category mat: (440134, 1661) nnz=1,296,424
  target distinct paths: 5,203  category mat: (181532, 1661) nnz=356,508
[3.3_construct]
  src_emb:
    items_total: 440134
    items_embedded: 5040
    items_placeholder: 435094
    cache_hits: 5040
    texts_truncated_this_run: 0
  tgt_emb:
    items_total: 181532
    items_embedded: 5342
    items_placeholder: 176190
    cache_hits: 5342
    texts_truncated_this_run: 0
  category_vocab: 1661


## §3.4 Integrate Data — *Merged Data*

- Within-vertical join: ratings ⨝ items on `parent_asin`. Coverage expected at
  100% per §2.4 — re-checked here on the cleaned set (spec §8).
- Cross-vertical user join: persist `shared_users` with per-domain activity
  counts — this is the supervised training set for the EMCDR mapping function.


In [17]:
# join coverage on the cleaned set — spec §8 'share of interactions whose
# item has metadata (expect 100%)'.
src_item_set = set(src_items['parent_asin'])
tgt_item_set = set(tgt_items['parent_asin'])
src_join_cov = float(src_5c['parent_asin'].isin(src_item_set).mean())
tgt_join_cov = float(tgt_5c['parent_asin'].isin(tgt_item_set).mean())
print(f'  source ratings⨝meta coverage: {src_join_cov*100:.4f}%')
print(f'  target ratings⨝meta coverage: {tgt_join_cov*100:.4f}%')

# shared-users with per-domain activity (positives only; the mapping function
# is trained on the user's positive behaviour in each domain)
def user_activity(signal: pd.DataFrame, users: Set[str], side: str) -> pd.DataFrame:
    pos = signal[(signal['label'] == 1) & (signal['user_id'].isin(users))]
    g = pos.groupby('user_id').agg(
        pos_count=('parent_asin', 'count'),
        first_ts=('timestamp', 'min'),
        last_ts=('timestamp', 'max'),
    )
    g.columns = [f'{side}_{c}' for c in g.columns]
    return g

src_act = user_activity(src_signal, shared_users, 'src')
tgt_act = user_activity(tgt_signal, shared_users, 'tgt')
shared_df = src_act.join(tgt_act, how='inner').reset_index().rename(columns={'user_id': 'user_id'})
print(f'  shared-user mapping rows: {len(shared_df):,}')

STATS['integrate'] = {
    'shared_users_post_5core': int(len(shared_users)),
    'shared_users_with_positive_activity_both_sides': int(len(shared_df)),
    'jaccard_post_5core': round(len(shared_users) / len(src_users | tgt_users), 4),
    'join_coverage': {
        'source': round(src_join_cov, 6),
        'target': round(tgt_join_cov, 6),
    },
}
log_stage('3.4_integrate', **STATS['integrate'])


  source ratings⨝meta coverage: 100.0000%
  target ratings⨝meta coverage: 100.0000%


  shared-user mapping rows: 108,936
[3.4_integrate]
  shared_users_post_5core: 109206
  shared_users_with_positive_activity_both_sides: 108936
  jaccard_post_5core: 0.0933
  join_coverage:
    source: 1.0
    target: 1.0


## §3.5 Format Data — *Reformatted Data*

- **ID remap** — contiguous integer `user_idx` and per-domain `item_idx`.
  Shared user vocab across domains so a shared user has one `user_idx`.
- **Splits** — temporal global cut by `split_ratios`, sorted on `timestamp`,
  with the two cutoff timestamps recorded for the report (spec §8).
- **Serialize** — Parquet for tables, NPZ for dense matrices, `meta.json` for
  the run, `stats.json` + `STATS.md` for the §3 reporting numbers.


In [18]:
# build unified user map (shared across domains), per-domain item maps
all_users = sorted(set(src_signal['user_id']).union(tgt_signal['user_id']))
user2idx = {u: i for i, u in enumerate(all_users)}

src_asins_kept = src_items['parent_asin'].tolist()
tgt_asins_kept = tgt_items['parent_asin'].tolist()
src_item2idx = {a: i for i, a in enumerate(src_asins_kept)}
tgt_item2idx = {a: i for i, a in enumerate(tgt_asins_kept)}

def remap(signal: pd.DataFrame, item2idx: Dict[str, int], name: str) -> pd.DataFrame:
    signal = signal.copy()
    signal['user_idx'] = signal['user_id'].map(user2idx).astype('int64')
    signal['item_idx'] = signal['parent_asin'].map(item2idx)
    drop = signal['item_idx'].isna().sum()
    if drop:
        print(f'  {name}: dropping {drop:,} signal rows referencing non-kept items '
              f'(sampled-negative pool may include them)')
        signal = signal.dropna(subset=['item_idx'])
    signal['item_idx'] = signal['item_idx'].astype('int64')
    return signal

src_out = remap(src_signal, src_item2idx, 'source')
tgt_out = remap(tgt_signal, tgt_item2idx, 'target')
print(f'  source rows after remap: {len(src_out):,}')
print(f'  target rows after remap: {len(tgt_out):,}')
print(f'  unified user vocab: {len(user2idx):,}  '
      f'(source items: {len(src_item2idx):,}, target items: {len(tgt_item2idx):,})')


  source rows after remap: 38,385,389
  target rows after remap: 29,341,642
  unified user vocab: 1,170,660  (source items: 440,134, target items: 181,532)


In [19]:
def assign_split(df: pd.DataFrame, ratios: Tuple[float, float, float], mode: str,
                  seed: int) -> Tuple[pd.Series, Dict]:
    """Returns (split_series, cutoffs_dict)."""
    df = df.copy()
    n = len(df)
    train_end = int(ratios[0] * n)
    val_end = int((ratios[0] + ratios[1]) * n)
    out = np.array(['train'] * n, dtype=object)
    cutoffs: Dict = {}
    if mode == 'temporal':
        order = np.argsort(df['timestamp'].values, kind='mergesort')
        sorted_ts = df['timestamp'].values[order]
        cutoffs = {
            'train_end_ts_ms': int(sorted_ts[train_end - 1]) if train_end > 0 else None,
            'val_end_ts_ms':   int(sorted_ts[val_end - 1])   if val_end   > 0 else None,
            'train_end_date': str(pd.to_datetime(sorted_ts[train_end - 1], unit='ms').date())
                              if train_end > 0 else None,
            'val_end_date':   str(pd.to_datetime(sorted_ts[val_end - 1], unit='ms').date())
                              if val_end > 0 else None,
        }
    elif mode == 'random':
        rng = np.random.default_rng(seed)
        order = rng.permutation(n)
    else:
        raise ValueError(mode)
    out[order[train_end:val_end]] = 'val'
    out[order[val_end:]] = 'test'
    return pd.Series(out, index=df.index, name='split'), cutoffs

src_split, src_cutoffs = assign_split(src_out, CFG.split_ratios, CFG.split, CFG.seed)
tgt_split, tgt_cutoffs = assign_split(tgt_out, CFG.split_ratios, CFG.split, CFG.seed)
src_out['split'] = src_split
tgt_out['split'] = tgt_split
print('source splits:'); print(src_out['split'].value_counts())
print('target splits:'); print(tgt_out['split'].value_counts())
print(f'source cutoffs: {src_cutoffs}')
print(f'target cutoffs: {tgt_cutoffs}')


source splits:
split
train    30708311
val       3838539
test      3838539
Name: count, dtype: int64
target splits:


split
train    23473313
test      2934165
val       2934164
Name: count, dtype: int64
source cutoffs: {'train_end_ts_ms': 1567460790372, 'val_end_ts_ms': 1619150398561, 'train_end_date': '2019-09-02', 'val_end_date': '2021-04-23'}
target cutoffs: {'train_end_ts_ms': 1525148823048, 'val_end_ts_ms': 1581093407425, 'train_end_date': '2018-05-01', 'val_end_date': '2020-02-07'}


In [20]:
# serialize -------------------------------------------------------------------
INTER_COLS = ['user_idx', 'item_idx', 'label', 'weight', 'rating', 'timestamp', 'split']
src_out[INTER_COLS].to_parquet(PAIR_DIR / 'source_interactions.parquet', index=False)
tgt_out[INTER_COLS].to_parquet(PAIR_DIR / 'target_interactions.parquet', index=False)

shared_df_idx = shared_df.copy()
shared_df_idx['user_idx'] = shared_df_idx['user_id'].map(user2idx).astype('int64')
shared_df_idx[['user_idx', 'src_pos_count', 'tgt_pos_count',
               'src_first_ts', 'src_last_ts', 'tgt_first_ts', 'tgt_last_ts']
            ].to_parquet(PAIR_DIR / 'shared_users.parquet', index=False)

# seen sets — long-form parquet (user_idx, domain, item_idx)
def seen_to_long(seen: Dict[str, Set[str]], item2idx: Dict[str, int], domain: str) -> pd.DataFrame:
    rows = []
    for u, items_set in seen.items():
        ui = user2idx.get(u)
        if ui is None: continue
        for it in items_set:
            ii = item2idx.get(it)
            if ii is not None:
                rows.append((ui, domain, ii))
    return pd.DataFrame(rows, columns=['user_idx', 'domain', 'item_idx'])

seen_long = pd.concat([
    seen_to_long(src_seen, src_item2idx, 'source'),
    seen_to_long(tgt_seen, tgt_item2idx, 'target'),
], ignore_index=True)
seen_long.to_parquet(PAIR_DIR / 'seen_sets.parquet', index=False)
print(f'  seen_sets.parquet rows: {len(seen_long):,}')

# item features — npz with embeddings + category multi-hot + has_text
np.savez(
    PAIR_DIR / 'source_item_features.npz',
    text_emb=src_emb,
    has_text=src_items['has_text'].values.astype('bool'),
    text_source=np.array(src_items['text_source'].tolist(), dtype=object),
    cat_indptr=src_cat.indptr, cat_indices=src_cat.indices, cat_shape=np.array(src_cat.shape),
    item_idx_to_asin=np.array(src_asins_kept, dtype=object),
)
np.savez(
    PAIR_DIR / 'target_item_features.npz',
    text_emb=tgt_emb,
    has_text=tgt_items['has_text'].values.astype('bool'),
    text_source=np.array(tgt_items['text_source'].tolist(), dtype=object),
    cat_indptr=tgt_cat.indptr, cat_indices=tgt_cat.indices, cat_shape=np.array(tgt_cat.shape),
    item_idx_to_asin=np.array(tgt_asins_kept, dtype=object),
)

# id_maps
id_maps = PAIR_DIR / 'id_maps'
id_maps.mkdir(exist_ok=True)
pd.DataFrame({'user_id': list(user2idx.keys()), 'user_idx': list(user2idx.values())}
            ).to_parquet(id_maps / 'user.parquet', index=False)
pd.DataFrame({'parent_asin': src_asins_kept, 'item_idx': range(len(src_asins_kept))}
            ).to_parquet(id_maps / 'source_item.parquet', index=False)
pd.DataFrame({'parent_asin': tgt_asins_kept, 'item_idx': range(len(tgt_asins_kept))}
            ).to_parquet(id_maps / 'target_item.parquet', index=False)
pd.DataFrame({'category': list(CAT_VOCAB.keys()), 'cat_idx': list(CAT_VOCAB.values())}
            ).to_parquet(id_maps / 'category.parquet', index=False)

# capture format-stage stats
STATS['format'] = {
    'n_users': int(len(user2idx)),
    'n_items': {'source': int(len(src_item2idx)), 'target': int(len(tgt_item2idx))},
    'split_mode': CFG.split,
    'split_ratios': list(CFG.split_ratios),
    'source': {
        'rows': int(len(src_out)),
        'split_counts': {k: int(v) for k, v in src_out['split'].value_counts().items()},
        'cutoffs': src_cutoffs,
    },
    'target': {
        'rows': int(len(tgt_out)),
        'split_counts': {k: int(v) for k, v in tgt_out['split'].value_counts().items()},
        'cutoffs': tgt_cutoffs,
    },
}
log_stage('3.5_format', **{k: v for k, v in STATS['format'].items()
                            if k not in ('source', 'target')})

# meta.json — config + embedding details
with open(PAIR_DIR / 'meta.json', 'w') as fh:
    json.dump(RUN_META, fh, indent=2, default=str)


  seen_sets.parquet rows: 16,667,275


[3.5_format]
  n_users: 1170660
  n_items:
    source: 440134
    target: 181532
  split_mode: temporal
  split_ratios: [0.8, 0.1, 0.1]


## Reporting outputs — `stats.json` + `STATS.md`

Per spec §8 / §4, this run emits two reporting artifacts:

- **`stats.json`** — machine-readable, every §3 reporting number this pipeline
  computed (per-step counts, label distribution, fallback/placeholder split,
  shared-user overlap, split sizes, cutoff timestamps).
- **`STATS.md`** — human-readable mirror, formatted as the tables the HW2 §3
  write-up is built from.

The HW2 §3 sections can be written directly from `STATS.md` without re-reading
the notebook.


In [21]:
# stats.json -----------------------------------------------------------------
with open(PAIR_DIR / 'stats.json', 'w') as fh:
    json.dump(STATS, fh, indent=2, default=str)
print(f'  wrote {(PAIR_DIR / "stats.json").relative_to(PROJECT_ROOT)}')

# STATS.md — human-readable mirror -------------------------------------------
def fmt(n):
    if isinstance(n, float):
        return f'{n:,.4f}' if abs(n) < 1 else f'{n:,.2f}'
    if isinstance(n, int):
        return f'{n:,d}'
    return str(n)

lines = []
P = lines.append

P(f'# STATS — `{STATS["pair"]}`')
P('')
P(f'_Snapshot date: {STATS["snapshot_date"]}._')
P('')
P(f'Generated by `notebooks/data_prep.ipynb` per spec §8.')
P('')

# --- §3.1 Select ------------------------------------------------------------
P('## §3.1 Select Data')
P('')
P('| | source ({}) | target ({}) |'.format(CFG.source, CFG.target))
P('|---|---:|---:|')
for k in ('interactions_0core', 'users_0core', 'items_0core',
          'dropped_rating_0', 'dropped_rating_3', 'interactions_after_drop',
          'interactions_5core', 'users_5core', 'items_5core',
          'retention_pct_vs_0core', 'k_core'):
    s = STATS['select']['source'].get(k)
    t_ = STATS['select']['target'].get(k)
    P(f'| {k} | {fmt(s)} | {fmt(t_)} |')
P('')
P(f'**Shared users after k-core**: {fmt(STATS["integrate"]["shared_users_post_5core"])}  '
  f'(Jaccard {STATS["integrate"]["jaccard_post_5core"]})')
P('')

# --- §3.2 Clean -------------------------------------------------------------
P('## §3.2 Clean Data')
P('')
P('| | source | target |')
P('|---|---:|---:|')
for k in ('items_kept', 'text_primary', 'text_fallback', 'text_none_placeholder',
          'orphan_interactions', 'duplicate_pairs'):
    s = STATS['clean']['source'].get(k)
    t_ = STATS['clean']['target'].get(k)
    P(f'| {k} | {fmt(s)} | {fmt(t_)} |')
P('')
P('### Per-field missing-value rate on kept items')
P('')
P('| field | source | target |')
P('|---|---:|---:|')
for fld in TRACKED_FIELDS:
    s = STATS['clean']['source']['missing_value_rates'].get(fld, 0)
    t_ = STATS['clean']['target']['missing_value_rates'].get(fld, 0)
    P(f'| `{fld}` | {s*100:.2f}% | {t_*100:.2f}% |')
P('')

# --- §3.3 Construct ---------------------------------------------------------
P('## §3.3 Construct Data')
P('')
P('### Label distribution')
P('')
P('| | source | target |')
P('|---|---:|---:|')
for k in ('positives', 'explicit_negatives', 'dropped_3', 'sampled_negatives'):
    s = STATS['construct']['source']['label_distribution'][k]
    t_ = STATS['construct']['target']['label_distribution'][k]
    P(f'| {k} | {fmt(s)} | {fmt(t_)} |')
P('')
P('### Embedding')
P('')
P(f'- Model: `{CFG.embed_model}`, dim={CFG.embed_dim}, '
  f'max_seq_length={CFG.max_seq_length}, L2-normalize={CFG.embed_normalize}')
P('')
P('| | source | target |')
P('|---|---:|---:|')
for k in ('items_total', 'items_embedded', 'items_placeholder', 'cache_hits',
          'texts_truncated_this_run'):
    s = STATS['construct']['source']['embedding'][k]
    t_ = STATS['construct']['target']['embedding'][k]
    P(f'| {k} | {fmt(s)} | {fmt(t_)} |')
P('')
P('### Category feature')
P('')
P(f'- Shared category vocab size: {fmt(STATS["construct"]["category_vocab_size"])}')
P(f'- Distinct category paths (source): {fmt(STATS["construct"]["source"]["category_distinct_paths"])}')
P(f'- Distinct category paths (target): {fmt(STATS["construct"]["target"]["category_distinct_paths"])}')
P('')

# --- §3.4 Integrate ---------------------------------------------------------
P('## §3.4 Integrate Data')
P('')
P(f'- Shared users after 5-core: {fmt(STATS["integrate"]["shared_users_post_5core"])}')
P(f'- Shared users w/ positive activity on both sides: '
  f'{fmt(STATS["integrate"]["shared_users_with_positive_activity_both_sides"])}')
P(f'- Join coverage source: {STATS["integrate"]["join_coverage"]["source"]*100:.4f}%')
P(f'- Join coverage target: {STATS["integrate"]["join_coverage"]["target"]*100:.4f}%')
P('')

# --- §3.5 Format ------------------------------------------------------------
P('## §3.5 Format Data')
P('')
P(f'- Unified user vocab: {fmt(STATS["format"]["n_users"])}')
P(f'- Source item vocab: {fmt(STATS["format"]["n_items"]["source"])}')
P(f'- Target item vocab: {fmt(STATS["format"]["n_items"]["target"])}')
P(f'- Split mode: {STATS["format"]["split_mode"]}, ratios={STATS["format"]["split_ratios"]}')
P('')
P('### Split counts')
P('')
P('| | source | target |')
P('|---|---:|---:|')
for sp in ('train', 'val', 'test'):
    s = STATS['format']['source']['split_counts'].get(sp, 0)
    t_ = STATS['format']['target']['split_counts'].get(sp, 0)
    P(f'| {sp} | {fmt(s)} | {fmt(t_)} |')
P('')
P('### Temporal-split cutoff timestamps')
P('')
src_co = STATS['format']['source']['cutoffs']
tgt_co = STATS['format']['target']['cutoffs']
P('| | source | target |')
P('|---|---:|---:|')
P(f'| train end | {src_co.get("train_end_date")} | {tgt_co.get("train_end_date")} |')
P(f'| val end   | {src_co.get("val_end_date")}   | {tgt_co.get("val_end_date")}   |')
P('')

with open(PAIR_DIR / 'STATS.md', 'w') as fh:
    fh.write('\n'.join(lines))
print(f'  wrote {(PAIR_DIR / "STATS.md").relative_to(PROJECT_ROOT)}')

# final artifact listing
print('\n=== ARTIFACTS ===')
for p in sorted(PAIR_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.stat().st_size/1e6:9.2f} MB  {p.relative_to(PAIR_DIR)}')


  wrote data/processed/Books__Movies_and_TV/stats.json
  wrote data/processed/Books__Movies_and_TV/STATS.md

=== ARTIFACTS ===
       0.00 MB  STATS.md
       0.03 MB  id_maps/category.parquet
       6.36 MB  id_maps/source_item.parquet
       2.86 MB  id_maps/target_item.parquet
      35.28 MB  id_maps/user.parquet
       0.00 MB  meta.json
      91.89 MB  seen_sets.parquet
       4.95 MB  shared_users.parquet
     365.46 MB  source_interactions.parquet
     693.12 MB  source_item_features.npz
       0.00 MB  stats.json
     263.65 MB  target_interactions.parquet
     285.16 MB  target_item_features.npz


---

## Hand-off

The training loop should:

1. Load `source_interactions.parquet` and `target_interactions.parquet`, filter
   by `split` for train / val / test, and use `label` + `weight` directly.
2. Load `*_item_features.npz` and look items up by `item_idx`. `text_emb` is
   384-dim and (when `CFG.embed_normalize=True`) L2-normalized. `has_text == False`
   → zero placeholder. `text_source ∈ {primary, fallback, none}` tells you
   *how* the text was built.
3. Load `shared_users.parquet` to supervise the source→target mapping function
   (one training pair per row: the user's source embedding maps to their
   target embedding).
4. Apply `seen_sets.parquet` at inference time to exclude items the user has
   already interacted with from the candidate pool.

Every choice in this pipeline is logged in `meta.json` and every reporting
number is in `stats.json` / `STATS.md`. The HW2 §3 sections can be written
directly from `STATS.md`.

**Open items the spec flags as parameters, not commitments (spec §10):**

- the source→target pair (re-run end-to-end for Books→Toys_and_Games for
  comparison before locking);
- `explicit_neg_weight > 1.0` ablation;
- `k_core_overrides` if the overlap floor warns;
- `embed_sample_size = int(N)` for fast smoke runs during dev.
